# Expert Feedback Analysis for Publication

This notebook generates two tables for the ACM publication:
1. Expert Dimension Evaluation (for Pain Severity and Pain Disability only)
2. Expert Questionnaire Feedback (PCS, BPI-IS, TSK-11SV)

Data source: `pain_narratives_acm_202512` schema

In [1]:
# Standard library imports
import sys
from pathlib import Path 
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np
from sqlalchemy import text
from sqlmodel import Session

# Add src to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / 'src'))

# Import database manager
from pain_narratives.core.database import DatabaseManager

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Initialize database manager
db = DatabaseManager()

print("✅ Libraries imported successfully")
print(f"📁 Working directory: {notebook_dir}")
print(f"🕐 Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully
📁 Working directory: /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks
🕐 Analysis run: 2026-05-26 15:10:55


## Table 1: Expert Dimension Evaluation

Dimensions: Pain Severity and Pain Disability  
Feedback types: score_alignment, explanation_alignment, usage_intent  
Values: Mean (SD)

In [2]:
# Load expert dimension evaluation data (wide format from database)
dimension_query = """
SELECT 
    experiment_id,
    participant_id,
    narrative_hash,
    user_id,
    word_count,
    severidad_score_alignment,
    severidad_explanation_alignment,
    severidad_usage_intent,
    discapacidad_score_alignment,
    discapacidad_explanation_alignment,
    discapacidad_usage_intent
FROM ai_narratives_original.expert_dimension_evaluation
"""

with Session(db.engine) as session:
    df_dimension_wide = pd.read_sql(text(dimension_query), session.connection())

print(f"Total evaluation records loaded: {len(df_dimension_wide)}")
print(f"\nSample data:")
print(df_dimension_wide.head(3))

# Transform from wide to long format
# Map Spanish dimension names to English for publication
dimension_mapping = {
    'severidad': 'Pain Severity',
    'discapacidad': 'Pain Disability'
}

# Create long format data
records = []
for _, row in df_dimension_wide.iterrows():
    for dim_spanish, dim_english in dimension_mapping.items():
        for feedback_type in ['score_alignment', 'explanation_alignment', 'usage_intent']:
            col_name = f"{dim_spanish}_{feedback_type}"
            rating_text = row[col_name]
            
            if pd.notna(rating_text):
                # Map text ratings to numeric values (Likert 1-7 scale)
                rating_map = {
                    'Strongly Disagree': 1,
                    'Disagree': 2,
                    'Somewhat Disagree': 3,
                    'Neither Agree nor Disagree': 4,
                    'Somewhat Agree': 5,
                    'Agree': 6,
                    'Strongly Agree': 7
                }
                rating_numeric = rating_map.get(rating_text, None)
                
                records.append({
                    'experiment_id': row['experiment_id'],
                    'participant_id': row['participant_id'],
                    'user_id': row['user_id'],
                    'dimension_name': dim_english,
                    'feedback_type': feedback_type,
                    'rating_text': rating_text,
                    'rating_numeric': rating_numeric
                })

df_dimension = pd.DataFrame(records)

print(f"\n✅ Transformed to long format: {len(df_dimension)} ratings")
print(f"\nDimensions found:")
print(df_dimension['dimension_name'].value_counts())
print(f"\nFeedback types:")
print(df_dimension['feedback_type'].value_counts())
print(f"\nRating distribution:")
print(df_dimension['rating_numeric'].value_counts().sort_index())

Total evaluation records loaded: 26

Sample data:
   experiment_id  participant_id narrative_hash  user_id word_count  \
0            116             3.0           None       11       None   
1            128            11.0           None       10       None   
2            141            11.0           None       18       None   

  severidad_score_alignment severidad_explanation_alignment  \
0                                                             
1                                                             
2                                                             

  severidad_usage_intent discapacidad_score_alignment  \
0                                                       
1                                                       
2                                                       

  discapacidad_explanation_alignment discapacidad_usage_intent  
0                                                               
1                                                    

In [3]:
# Calculate statistics from the already-transformed long format data
# df_dimension already has: dimension_name, feedback_type, rating_numeric

# Calculate statistics grouped by dimension and feedback type
stats = df_dimension.groupby(['dimension_name', 'feedback_type'])['rating_numeric'].agg(
    mean='mean',
    std='std',
    count='count'
).reset_index()

# Format as Mean (SD)
stats['mean_sd'] = stats.apply(
    lambda row: f"{row['mean']:.2f} ({row['std']:.2f})", 
    axis=1
)

# Pivot to desired format: dimensions as rows, feedback_types as columns
table1 = stats.pivot(
    index='dimension_name',
    columns='feedback_type',
    values='mean_sd'
)

# Reorder columns
column_order = ['score_alignment', 'explanation_alignment', 'usage_intent']
table1 = table1[column_order]

# Rename columns for publication
table1.columns = ['Score Alignment', 'Explanation Alignment', 'Usage Intent']
table1.index.name = 'Dimension'

print("\nTable 1: Expert Dimension Evaluation")
print("="*70)
print(table1)
print("\nNote: Values are Mean (SD)")


Table 1: Expert Dimension Evaluation
                Score Alignment Explanation Alignment Usage Intent
Dimension                                                         
Pain Disability       nan (nan)             nan (nan)    nan (nan)
Pain Severity         nan (nan)             nan (nan)    nan (nan)

Note: Values are Mean (SD)


## Table 2: Expert Questionnaire Feedback

Questionnaires: PCS, BPI-IS, TSK-11SV  
Rating types: authenticity, reasoning_adequacy  
Values: Mean (SD) and Median [IQR]

In [4]:
# Load expert questionnaire feedback data
questionnaire_query = """
SELECT 
    questionnaire_id,
    participant_id,
    narrative_hash,
    user_id,
    questionnaire_name,
    authenticity_rating,
    reasoning_adequacy_rating,
    word_count
FROM ai_narratives_original.expert_questionnaire_feedback
"""

with Session(db.engine) as session:
    df_questionnaire_wide = pd.read_sql(text(questionnaire_query), session.connection())

print(f"Total questionnaire feedback records loaded: {len(df_questionnaire_wide)}")
print(f"\nSample data:")
print(df_questionnaire_wide.head(3))

# Transform from wide to long format
records_quest = []
for _, row in df_questionnaire_wide.iterrows():
    for rating_type in ['authenticity', 'reasoning_adequacy']:
        col_name = f"{rating_type}_rating"
        rating_text = row[col_name]
        
        if pd.notna(rating_text):
            # Map text ratings to numeric values (Likert 1-7 scale)
            rating_map = {
                'Strongly Disagree': 1,
                'Disagree': 2,
                'Somewhat Disagree': 3,
                'Neither Agree nor Disagree': 4,
                'Somewhat Agree': 5,
                'Agree': 6,
                'Strongly Agree': 7
            }
            rating_numeric = rating_map.get(rating_text, None)
            
            records_quest.append({
                'questionnaire_id': row['questionnaire_id'],
                'participant_id': row['participant_id'],
                'user_id': row['user_id'],
                'questionnaire_name': row['questionnaire_name'],
                'rating_type': rating_type,
                'rating_text': rating_text,
                'rating_numeric': rating_numeric
            })

df_questionnaire = pd.DataFrame(records_quest)

print(f"\n✅ Transformed to long format: {len(df_questionnaire)} ratings")
print(f"\nQuestionnaires found:")
print(df_questionnaire['questionnaire_name'].value_counts())
print(f"\nRating types:")
print(df_questionnaire['rating_type'].value_counts())
print(f"\nRating distribution:")
print(df_questionnaire['rating_numeric'].value_counts().sort_index())

Total questionnaire feedback records loaded: 70

Sample data:
   questionnaire_id  participant_id narrative_hash  user_id  \
0               189             3.0           None       11   
1               190             3.0           None       11   
2               191             3.0           None       11   

  questionnaire_name authenticity_rating   reasoning_adequacy_rating  \
0                PCS      Somewhat Agree                       Agree   
1             BPI-IS               Agree  Neither Agree Nor Disagree   
2           TSK-11SV   Somewhat Disagree              Somewhat Agree   

  word_count  
0       None  
1       None  
2       None  

✅ Transformed to long format: 140 ratings

Questionnaires found:
questionnaire_name
BPI-IS      48
PCS         46
TSK-11SV    46
Name: count, dtype: int64

Rating types:
rating_type
authenticity          70
reasoning_adequacy    70
Name: count, dtype: int64

Rating distribution:
rating_numeric
2.0     1
3.0     4
5.0    47
6.0    70


In [5]:
# Calculate statistics from the already-transformed long format data
# df_questionnaire already has: questionnaire_name, rating_type, rating_numeric

# Calculate statistics grouped by questionnaire and rating type
quest_stats = df_questionnaire.groupby(['questionnaire_name', 'rating_type'])['rating_numeric'].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('median', 'median'),
    ('q25', lambda x: x.quantile(0.25)),
    ('q75', lambda x: x.quantile(0.75)),
    ('count', 'count')
]).reset_index()

# Format Mean (SD)
quest_stats['mean_sd'] = quest_stats.apply(
    lambda row: f"{row['mean']:.2f} ({row['std']:.2f})", 
    axis=1
)

# Format Median [IQR]
quest_stats['median_iqr'] = quest_stats.apply(
    lambda row: f"{row['median']:.2f} [{row['q25']:.2f}-{row['q75']:.2f}]", 
    axis=1
)

# Create final table
table2 = quest_stats[['questionnaire_name', 'rating_type', 'mean_sd', 'median_iqr']].copy()

# Rename columns for publication
table2.columns = ['Questionnaire', 'Rating Type', 'Mean (SD)', 'Median [IQR]']

# Sort by questionnaire and rating type
table2 = table2.sort_values(['Questionnaire', 'Rating Type']).reset_index(drop=True)

print("\nTable 2: Expert Questionnaire Feedback")
print("="*100)
print(table2.to_string(index=False))


Table 2: Expert Questionnaire Feedback
Questionnaire        Rating Type   Mean (SD)     Median [IQR]
       BPI-IS       authenticity 5.54 (0.83) 6.00 [5.00-6.00]
       BPI-IS reasoning_adequacy 5.52 (0.67) 5.00 [5.00-6.00]
          PCS       authenticity 5.77 (0.61) 6.00 [5.00-6.00]
          PCS reasoning_adequacy 5.91 (0.67) 6.00 [5.50-6.00]
     TSK-11SV       authenticity 5.48 (1.12) 6.00 [5.00-6.00]
     TSK-11SV reasoning_adequacy 5.55 (1.01) 6.00 [5.00-6.00]


## Export Tables for Publication

In [6]:
# Save tables to CSV for easy import into publication document
output_dir = notebook_dir / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

# Export tables
table1.to_csv(output_dir / '03_table_expert_dimension_evaluation.csv')
table2.to_csv(output_dir / '03_table_expert_questionnaire_feedback.csv', index=False)

print("Tables exported successfully!")
print(f"- {output_dir / '03_table_expert_dimension_evaluation.csv'}")
print(f"- {output_dir / '03_table_expert_questionnaire_feedback.csv'}")

Tables exported successfully!
- /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/03_table_expert_dimension_evaluation.csv
- /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/03_table_expert_questionnaire_feedback.csv


## Summary Statistics

In [7]:
print("Summary Statistics")
print("="*70)
print(f"\nDimension Evaluations:")
print(f"  - Total evaluations: {len(df_dimension)}")
print(f"  - Dimensions: {df_dimension['dimension_name'].nunique()}")
print(f"  - Pain Severity evaluations: {len(df_dimension[df_dimension['dimension_name'] == 'Pain Severity'])}")
print(f"  - Pain Disability evaluations: {len(df_dimension[df_dimension['dimension_name'] == 'Pain Disability'])}")

print(f"\nQuestionnaire Feedback:")
print(f"  - Total feedbacks: {len(df_questionnaire)}")
print(f"  - Questionnaires: {df_questionnaire['questionnaire_name'].nunique()}")
for q in df_questionnaire['questionnaire_name'].unique():
    count = len(df_questionnaire[df_questionnaire['questionnaire_name'] == q])
    print(f"  - {q}: {count} feedbacks")

Summary Statistics

Dimension Evaluations:
  - Total evaluations: 156
  - Dimensions: 2
  - Pain Severity evaluations: 78
  - Pain Disability evaluations: 78

Questionnaire Feedback:
  - Total feedbacks: 140
  - Questionnaires: 3
  - PCS: 46 feedbacks
  - BPI-IS: 48 feedbacks
  - TSK-11SV: 46 feedbacks
